# X-REACH Full Preprocessing Pipeline — Data Preprocessing → Cross-Validation → RL Problem Formulation (Augmented)

This is a single, consolidated notebook that reproduces your **entire
original preprocessing pipeline** end-to-end:

```
heart_disease_uci.csv (raw, 920 rows)
  -> feature_selection.py technique          [Stage 1]
heart_disease_trimmed.csv (539 rows, 10 raw features + target)
  -> fit_preprocessing_params.py's RobustScaler + one-hot technique   [Stage 2]
preprocessed_dataset (539 rows, 15-dim + target)
  -> StratifiedKFold(5), same as CrossValidation.ipynb                [Stage 3]
  -> per-fold TRAINING-ONLY augmentation (NEW, replaces undersampling) [Stage 4]
  -> RL_Problem_Formulation.ipynb technique                           [Stage 5]
rl_environment.pkl (5 folds, ready to feed straight into your existing
                     DQN / PPO training notebooks unchanged)
```

It was built by directly reading your own notebooks/scripts
(`research/preprocessing/feature_selection.py`,
`research/preprocessing/fit_preprocessing_params.py`,
`research/preprocessing/CrossValidation.ipynb`,
`research/preprocessing/Dataset_Balancing.ipynb`,
`research/preprocessing/RL_Problem_Formulation.ipynb`) and porting each
stage's actual technique. Two things are **deliberately changed**, both
explained in the markdown cells right before where they happen:

## Change 1 — a real bug found in the original pipeline, now fixed

`CrossValidation.ipynb` computes its 5-fold split on
`preprocessed_dataset_balanced_*.csv` (418 rows, after `Dataset_Balancing.ipynb`
undersamples the majority class down to match the minority class). It saves
only the **row *positions*** (`train_idx`/`test_idx`, ranging 0-417) to
`cross_validation_folds.pkl`.

`RL_Problem_Formulation.ipynb` then loads the **original, unbalanced**
`preprocessed_dataset_2026-07-04_07-18-06.csv` (539 rows) and applies those
same position indices to it via `.iloc[train_idx]`.

Verified directly against your own saved artifacts in this repo:
`cross_validation_folds.pkl`'s indices top out at **417**, and the row
counts inside the already-exported `rl_environment.pkl` sum to exactly
**418** per split family, not 539. That means every fold was actually built
from **418 arbitrary row positions of the 539-row file** — not the patients
the split was computed for — and **121 real patients (539-418) were never
used at all**, silently, with no error. This notebook fixes it by
splitting and augmenting the *same* dataframe throughout, so the indices
always mean what they say.

## Change 2 — undersampling replaced with augmentation

The original `Dataset_Balancing.ipynb` balances classes by **discarding**
real rows from the majority class down to the minority class's count
(539 → 418 rows) — the opposite of what you asked for. Stage 4 below
replaces that with clinically-safe **oversampling by interpolation**,
applied only to each fold's training split (never its test split, to avoid
leakage — see Stage 4's markdown for why that ordering matters), which
balances classes by **growing** the minority class instead of shrinking the
majority one.

Everything else (the RobustScaler formula, the one-hot column order, the
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` call, and the
final `rl_environment.pkl` schema) is your original technique, unchanged.


## Setup

In [ ]:
import os
import json
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


## Stage 1 — Feature selection & cleaning

Ports `research/preprocessing/feature_selection.py` exactly: selects the 10
canonical columns + `num` from the raw UCI file, renames `thalch` ->
`thalach`, encodes every categorical column to the same integer codes used
throughout the app, drops rows with missing values in the 10 canonical
columns, and binarizes `num` into `target`.

Upload **either**:
- `heart_disease_uci.csv` (raw, 920 rows) to run this stage, **or**
- `heart_disease_trimmed.csv` (already cleaned, 539 rows) to skip straight
  to Stage 2.


In [ ]:
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs",
]
SOURCE_COLUMN_MAP = {"thalach": "thalch"}
SEX_MAP = {"Male": 1, "Female": 0}
BOOL_MAP = {True: 1, False: 0}
SLOPE_MAP = {"downsloping": 0, "flat": 1, "upsloping": 2}
CP_MAP = {"typical angina": 0, "atypical angina": 1, "non-anginal": 2, "asymptomatic": 3}
RESTECG_MAP = {"normal": 0, "st-t abnormality": 1, "lv hypertrophy": 2}


def clean_raw_dataset(raw):
    """Faithful port of feature_selection.py::load_and_clean()."""
    subset = raw[[SOURCE_COLUMN_MAP.get(c, c) for c in CANONICAL_FEATURE_ORDER] + ["num"]].copy()
    subset.columns = CANONICAL_FEATURE_ORDER + ["num"]

    before = len(subset)
    subset = subset.dropna(subset=CANONICAL_FEATURE_ORDER).reset_index(drop=True)
    print(f"Rows before dropping missing values: {before}")
    print(f"Rows after dropping missing values in the 10 canonical columns: {len(subset)}")

    subset["sex"] = subset["sex"].map(SEX_MAP)
    subset["fbs"] = subset["fbs"].map(BOOL_MAP)
    subset["exang"] = subset["exang"].map(BOOL_MAP)
    subset["slope"] = subset["slope"].map(SLOPE_MAP)
    subset["cp"] = subset["cp"].map(CP_MAP)
    subset["restecg"] = subset["restecg"].map(RESTECG_MAP)

    encoded_cols = ["sex", "fbs", "exang", "slope", "cp", "restecg"]
    if subset[encoded_cols].isna().any().any():
        raise ValueError("Unexpected category value -- check the *_MAP dictionaries above.")
    for col in encoded_cols:
        subset[col] = subset[col].astype(int)

    subset["target"] = (subset["num"] > 0).astype(int)
    return subset.drop(columns=["num"])[CANONICAL_FEATURE_ORDER + ["target"]]


TRIMMED_PATH = "heart_disease_trimmed.csv"
RAW_PATH = "heart_disease_uci.csv"

if not os.path.exists(TRIMMED_PATH) and not os.path.exists(RAW_PATH):
    try:
        from google.colab import files
        print("Upload heart_disease_uci.csv OR heart_disease_trimmed.csv")
        uploaded = files.upload()
        for name in uploaded:
            if "trimmed" in name:
                TRIMMED_PATH = name
            else:
                RAW_PATH = name
    except ImportError:
        raise FileNotFoundError("Place heart_disease_uci.csv or heart_disease_trimmed.csv next to this notebook.")

if os.path.exists(TRIMMED_PATH):
    trimmed = pd.read_csv(TRIMMED_PATH)
    print(f"Loaded already-trimmed dataset: {len(trimmed)} rows (skipped Stage 1 cleaning)")
else:
    raw = pd.read_csv(RAW_PATH)
    trimmed = clean_raw_dataset(raw)
    trimmed.to_csv("heart_disease_trimmed.csv", index=False)
    print(f"Cleaned {len(trimmed)} rows -> heart_disease_trimmed.csv")

print(trimmed["target"].value_counts())
trimmed.head()


## Stage 2 — RobustScaler + one-hot encoding

Ports `research/preprocessing/fit_preprocessing_params.py` exactly: fits
`(raw - median) / IQR` on the four continuous columns **across the whole
539-row trimmed dataset** (matching the already-deployed
`models/rl/preprocessing_params.json` — this global fit, rather than a
per-fold refit, is a deliberate choice to keep this pipeline's output
compatible with the currently-running backend without any backend changes
or database reseeding), then one-hot expands `cp`/`restecg` into the fixed
15-dimension column order already in production use.


In [ ]:
CONTINUOUS_COLUMNS = ["age", "trestbps", "thalach", "oldpeak"]
CP_INT_TO_ONEHOT = {0: "cp_typical angina", 1: "cp_atypical angina", 2: "cp_non-anginal", 3: "cp_asymptomatic"}
RESTECG_INT_TO_ONEHOT = {0: "restecg_normal", 1: "restecg_st-t abnormality", 2: "restecg_lv hypertrophy"}
TRAINED_FEATURE_ORDER_15DIM = [
    "sex", "fbs", "exang", "age", "trestbps", "thalach", "oldpeak",
    "cp_asymptomatic", "cp_atypical angina", "cp_non-anginal", "cp_typical angina",
    "restecg_lv hypertrophy", "restecg_normal", "restecg_st-t abnormality",
    "slope",
]


def fit_robust_scaler_params(df):
    """Faithful port of fit_preprocessing_params.py::fit_robust_scaler_params()."""
    params = {}
    for col in CONTINUOUS_COLUMNS:
        values = df[col].astype(float)
        median = float(values.median())
        q1, q3 = float(values.quantile(0.25)), float(values.quantile(0.75))
        params[col] = {"median": median, "iqr": q3 - q1}
    return params


SCALER_PARAMS = fit_robust_scaler_params(trimmed)
print("Fitted scaler params (should match models/rl/preprocessing_params.json):")
print(json.dumps(SCALER_PARAMS, indent=2))


def preprocess_row(row):
    """Raw canonical feature row -> 15-dim vector. Same formula as
    backend/services/preprocessingService.js::toTrainedRepresentation."""
    vector = {
        "sex": row["sex"], "fbs": row["fbs"], "exang": row["exang"],
        "age": (row["age"] - SCALER_PARAMS["age"]["median"]) / SCALER_PARAMS["age"]["iqr"],
        "trestbps": (row["trestbps"] - SCALER_PARAMS["trestbps"]["median"]) / SCALER_PARAMS["trestbps"]["iqr"],
        "thalach": (row["thalach"] - SCALER_PARAMS["thalach"]["median"]) / SCALER_PARAMS["thalach"]["iqr"],
        "oldpeak": (row["oldpeak"] - SCALER_PARAMS["oldpeak"]["median"]) / SCALER_PARAMS["oldpeak"]["iqr"],
        "cp_asymptomatic": 0, "cp_atypical angina": 0, "cp_non-anginal": 0, "cp_typical angina": 0,
        "restecg_lv hypertrophy": 0, "restecg_normal": 0, "restecg_st-t abnormality": 0,
        "slope": row["slope"],
    }
    vector[CP_INT_TO_ONEHOT[int(row["cp"])]] = 1
    vector[RESTECG_INT_TO_ONEHOT[int(row["restecg"])]] = 1
    return vector


preprocessed_rows = [preprocess_row(r) for _, r in trimmed.iterrows()]
preprocessed = pd.DataFrame(preprocessed_rows, columns=TRAINED_FEATURE_ORDER_15DIM)
preprocessed["target"] = trimmed["target"].values

STATE_SIZE = len(TRAINED_FEATURE_ORDER_15DIM)
print(f"\nPreprocessed dataset: {preprocessed.shape[0]} rows x {STATE_SIZE} features")
preprocessed.head()


## Stage 3 — 5-Fold Stratified Cross-Validation

Same technique as `CrossValidation.ipynb`
(`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`), but run
directly on `preprocessed` (539 rows) — **not** on a separately-balanced,
differently-sized, differently-ordered dataframe. This is the fix for the
index-mismatch bug described at the top of this notebook: the indices
computed here are used against this exact same dataframe in every stage
that follows, so they always point at the patient they were computed for.


In [ ]:
X = preprocessed.drop("target", axis=1)
y = preprocessed["target"]

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
raw_fold_indices = list(kfold.split(X, y))

for i, (train_idx, test_idx) in enumerate(raw_fold_indices):
    print(f"Fold {i+1}: {len(train_idx)} train / {len(test_idx)} test "
          f"(train positive rate {y.iloc[train_idx].mean():.3f}, "
          f"test positive rate {y.iloc[test_idx].mean():.3f})")


## Stage 4 — Per-fold training-only augmentation (replaces undersampling)

For each fold, **only the training split** is augmented — the test split
stays exactly the real rows `StratifiedKFold` assigned to it. This ordering
matters for medical data specifically: augmenting *before* splitting risks
a synthetic row derived from what becomes a test-set patient leaking into
that same fold's training data, which would inflate the reported accuracy
without the model actually generalizing better.

Synthetic rows are generated by **interpolating between two real patients
who already share the same diagnosis label** — continuous vitals
(`age`, `trestbps`, `thalach`, `oldpeak`, already RobustScaled at this
point) are a random point on the line between a real anchor patient and one
of its same-class nearest neighbors, clipped to the range actually observed
among real patients for that feature (so a synthetic row can never exceed
what a real patient in the data exhibited). The categorical/one-hot block
(`sex`, `fbs`, `exang`, `slope`, and the `cp_*`/`restecg_*` one-hot columns)
is copied **verbatim** from the anchor patient as a single unit — this both
avoids fabricating a symptom combination no real patient has, and keeps the
one-hot columns automatically valid (exactly one `cp_*` and one `restecg_*`
column set).

The augmentation below guarantees each fold's **final training set has an
exactly equal number of heart-disease and no-heart-disease cases** (per
your decision): it first oversamples the minority class up to the majority
class's count, then grows *both* classes by the same additional amount
(`GROWTH_FACTOR`), so the two classes always end up perfectly balanced, not
just approximately -- asserted explicitly at the end of the cell below.


In [ ]:
CONTINUOUS_FEATURES = ["age", "trestbps", "thalach", "oldpeak"]
N_NEIGHBORS = 5
GROWTH_FACTOR = 1.0   # extra synthetic rows added to EACH class, as a fraction of that class's (post-balancing) size


def augment_training_fold(train_df, seed):
    """train_df: real rows only, already 15-dim + target.
    Returns real rows + synthetic rows: first balances classes by
    oversampling the minority class to match the majority, then grows both
    classes further by GROWTH_FACTOR.
    """
    rng = np.random.RandomState(seed)
    class_counts = train_df["target"].value_counts()
    majority_count = int(class_counts.max())

    synthetic_rows = []
    for label in class_counts.index:
        class_df = train_df[train_df["target"] == label].reset_index(drop=True)
        n_to_balance = majority_count - len(class_df)
        n_growth = int(round((len(class_df) + max(n_to_balance, 0)) * GROWTH_FACTOR))
        n_synthetic = max(n_to_balance, 0) + n_growth
        if n_synthetic == 0 or len(class_df) < 2:
            continue

        k = min(N_NEIGHBORS, len(class_df) - 1)
        cont = class_df[CONTINUOUS_FEATURES].to_numpy(dtype=np.float64)
        cont_min, cont_max = cont.min(axis=0), cont.max(axis=0)   # real-data range, used to clip synthetic values
        cont_mean, cont_std = cont.mean(axis=0), cont.std(axis=0) + 1e-8
        cont_norm = (cont - cont_mean) / cont_std

        nn_model = NearestNeighbors(n_neighbors=k + 1).fit(cont_norm)
        _, neighbor_idx = nn_model.kneighbors(cont_norm)

        for _ in range(n_synthetic):
            anchor_i = rng.randint(0, len(class_df))
            neighbor_choices = neighbor_idx[anchor_i, 1:]
            neighbor_i = neighbor_choices[rng.randint(0, len(neighbor_choices))]

            anchor_row = class_df.iloc[anchor_i]
            neighbor_row = class_df.iloc[neighbor_i]

            synthetic = anchor_row.copy()
            t = rng.uniform(0.0, 1.0)
            for j, feat in enumerate(CONTINUOUS_FEATURES):
                interpolated = anchor_row[feat] + t * (neighbor_row[feat] - anchor_row[feat])
                synthetic[feat] = float(np.clip(interpolated, cont_min[j], cont_max[j]))
            synthetic_rows.append(synthetic)

    if not synthetic_rows:
        return train_df.copy()

    synthetic_df = pd.DataFrame(synthetic_rows).reset_index(drop=True)
    augmented = pd.concat([train_df, synthetic_df], ignore_index=True)
    return augmented.sample(frac=1.0, random_state=seed).reset_index(drop=True)


fold_summaries = []
augmented_folds = []
for i, (train_idx, test_idx) in enumerate(raw_fold_indices):
    train_real = preprocessed.iloc[train_idx].reset_index(drop=True)
    test_real = preprocessed.iloc[test_idx].reset_index(drop=True)

    train_augmented = augment_training_fold(train_real, seed=SEED + i)

    augmented_folds.append({
        "fold": i + 1,
        "train": train_augmented,
        "test": test_real,
        "n_real_train": len(train_real),
        "n_synthetic_train": len(train_augmented) - len(train_real),
    })

    counts = train_augmented["target"].value_counts()
    assert counts.nunique() == 1, (
        f"Fold {i+1}: classes are not equally balanced after augmentation ({dict(counts)}) -- "
        "this should never happen; check augment_training_fold()."
    )

    print(f"Fold {i+1}: {len(train_real)} real train rows "
          f"(class balance {dict(train_real['target'].value_counts())}) + "
          f"{len(train_augmented) - len(train_real)} synthetic = {len(train_augmented)} total "
          f"(class balance {dict(counts)} -- EQUAL, as required) | "
          f"{len(test_real)} real test rows (untouched)")


## Stage 5 — RL Problem Formulation

Same technique as `RL_Problem_Formulation.ipynb`: separate features/target,
take `.values` as the state array, `action_size = 2` (binary
classification). The only difference from the original is that the
augmented training array now includes the synthetic rows from Stage 4, and
every fold's `states_train`/`states_test` come from the *same* dataframe
the split indices were computed against (the bug fix).


In [ ]:
rl_environment = []

for fold in augmented_folds:
    X_train = fold["train"].drop(columns=["target"])
    y_train = fold["train"]["target"]
    X_test = fold["test"].drop(columns=["target"])
    y_test = fold["test"]["target"]

    rl_environment.append({
        "fold": fold["fold"],
        "states_train": X_train.values.astype(np.float32),
        "labels_train": y_train.values.astype(np.int64),
        "states_test": X_test.values.astype(np.float32),
        "labels_test": y_test.values.astype(np.int64),
        "state_size": X_train.shape[1],
        "action_size": 2,
        "augmentation": {
            "n_real_train": fold["n_real_train"],
            "n_synthetic_train": fold["n_synthetic_train"],
        },
    })

    print(f"Fold {fold['fold']}: state_size={rl_environment[-1]['state_size']}, "
          f"states_train={rl_environment[-1]['states_train'].shape}, "
          f"states_test={rl_environment[-1]['states_test'].shape}")


## Stage 6 — Save `rl_environment.pkl`

This is a **drop-in replacement** for the existing `rl_environment.pkl` —
upload this file to your existing `EHR_DQN_PyTorch.ipynb` / `PPO.ipynb`
exactly where they currently expect `rl_environment.pkl`, and nothing else
in those notebooks needs to change to pick up the augmented, correctly-split
training data.

If you also want the entropy-coefficient / dropout / weight-decay /
`output_activation` export fixes discussed earlier (for the saturation and
non-monotonic-age issues), those are separate, small edits to
`DQN.ipynb`/`PPO.ipynb` themselves — this notebook only fixes the data
pipeline feeding into them.


In [ ]:
OUTPUT_PATH = "rl_environment.pkl"
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(rl_environment, f)

# Also save the fitted scaler params + a manifest, for traceability.
manifest = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "source_trimmed_rows": len(trimmed),
    "source_preprocessed_rows": len(preprocessed),
    "scaler_params": SCALER_PARAMS,
    "n_folds": len(rl_environment),
    "growth_factor": GROWTH_FACTOR,
    "per_fold": [
        {"fold": f["fold"], "n_real_train": f["augmentation"]["n_real_train"],
         "n_synthetic_train": f["augmentation"]["n_synthetic_train"],
         "n_test_real": len(f["states_test"])}
        for f in rl_environment
    ],
}
with open("rl_environment_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH)} bytes)")
print("Saved rl_environment_manifest.json")

try:
    from google.colab import files
    files.download(OUTPUT_PATH)
    files.download("rl_environment_manifest.json")
except ImportError:
    print("Not in Colab -- files are available locally.")
